# Notebook 14 — Final CIC-IDS2017 Dataset Preparation

This notebook is the transition from **analysis to implementation**.

It consumes the raw CIC-IDS2017 CSV files and the final preprocessing specification from Notebook 13. It produces reproducible train/validation/test artifacts without modifying the raw dataset.

The implementation is chunk-aware and avoids concatenating all 2.8M rows into a single in-memory DataFrame unnecessarily.

## Colab setup

Recommended storage:

```text
/content/drive/MyDrive/CIC-Dataset-Analysis/
├── data/raw/cicids2017/
└── results/final_dataset_selection/
```

If the dataset is already in Google Drive, mount Drive. Otherwise upload a ZIP containing the raw CIC-IDS2017 CSVs.

The notebook never writes processed files back into `data/raw`.

In [14]:
from pathlib import Path

import gc, json, os, re, time, zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import joblib

# ============================================================
# Project paths
# ============================================================

REPO = Path("/content/drive/MyDrive/CIC-Dataset-Analysis")

SPEC_DIR = REPO / "results/final_dataset_selection"

# Kaggle dataset will be downloaded here automatically
RAW_DIR = Path("/content/cicids2017_raw")

OUT_DIR = REPO / "data/processed/cicids2017"
RESULTS_DIR = REPO / "results/cicids2017/14_final_preparation"

OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Raw dataset directory:", RAW_DIR)
print("Output directory:", OUT_DIR)
print("Results directory:", RESULTS_DIR)

Raw dataset directory: /content/cicids2017_raw
Output directory: /content/drive/MyDrive/CIC-Dataset-Analysis/data/processed/cicids2017
Results directory: /content/drive/MyDrive/CIC-Dataset-Analysis/results/cicids2017/14_final_preparation


In [16]:
# ============================================================
# Download CIC-IDS2017 directly from Kaggle
# ============================================================

from google.colab import userdata

KAGGLE_DATASET = "chethuhn/network-intrusion-dataset"

# Retrieve Kaggle API token from Colab Secrets
try:
    KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Could not access KAGGLE_API_TOKEN from Colab Secrets. "
        "Make sure the secret exists and is available to this notebook."
    ) from e

if not KAGGLE_API_TOKEN:
    raise ValueError("KAGGLE_API_TOKEN is empty.")

# Kaggle CLI reads this environment variable
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

print(f"Downloading Kaggle dataset: {KAGGLE_DATASET}")
print(f"Destination: {RAW_DIR}")

# Install Kaggle CLI
os.system("pip -q install kaggle")

# Download and extract directly into the Colab runtime
result = os.system(
    f'kaggle datasets download -d "{KAGGLE_DATASET}" '
    f'--path "{RAW_DIR}" --unzip'
)

if result != 0:
    raise RuntimeError(
        "Kaggle dataset download failed. "
        "Check the Kaggle token and dataset access."
    )

# Locate CSV files
csv_files = sorted(RAW_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files were found after downloading {KAGGLE_DATASET}."
    )

print(f"\nCSV files found: {len(csv_files)}")

for p in csv_files:
    print("-", p)

Destination: /content/cicids2017_raw

CSV files found: 8
- /content/cicids2017_raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
- /content/cicids2017_raw/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
- /content/cicids2017_raw/Friday-WorkingHours-Morning.pcap_ISCX.csv
- /content/cicids2017_raw/Monday-WorkingHours.pcap_ISCX.csv
- /content/cicids2017_raw/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
- /content/cicids2017_raw/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
- /content/cicids2017_raw/Tuesday-WorkingHours.pcap_ISCX.csv
- /content/cicids2017_raw/Wednesday-workingHours.pcap_ISCX.csv


## 1. Load the specification and inspect headers

Do not hard-code a feature list in this notebook unless Notebook 13 explicitly produced one. The specification is the source of truth.

In [22]:
# ============================================================
# Load preprocessing specification from Notebook 13
# ============================================================

SPEC_FILENAME = "preprocessing_specification.csv"

# Expected repository location
spec_path = REPO / "results" / "final_dataset_selection" / SPEC_FILENAME

print("Looking for preprocessing specification:")
print(spec_path)
print("Exists:", spec_path.exists())

# ------------------------------------------------------------
# Locate specification if REPO path differs
# ------------------------------------------------------------

if not spec_path.exists():
    search_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content"),
    ]

    matches = []

    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(SPEC_FILENAME))

    matches = list(dict.fromkeys(matches))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {SPEC_FILENAME} anywhere under "
            f"/content/drive/MyDrive or /content."
        )

    print("\nFound specification at:")
    for match in matches:
        print("-", match)

    spec_path = matches[0]

print("\nUsing:")
print(spec_path)

# ------------------------------------------------------------
# Load preprocessing specification
# ------------------------------------------------------------

spec = pd.read_csv(spec_path)

print("\nPreprocessing specification:")
display(spec)

spec_dict = dict(zip(spec["parameter"], spec["value"]))

# ------------------------------------------------------------
# Extract preprocessing configuration
# ------------------------------------------------------------

TARGET_COLUMN = spec_dict["target_column"].strip()

RANDOM_STATE = int(spec_dict["random_state"])

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

print("\nResolved configuration:")
print("Target column :", repr(TARGET_COLUMN))
print("Random state  :", RANDOM_STATE)
print("Train fraction:", TRAIN_FRAC)
print("Validation    :", VAL_FRAC)
print("Test fraction :", TEST_FRAC)

# ------------------------------------------------------------
# Read dataset header
# ------------------------------------------------------------

header = pd.read_csv(csv_files[0], nrows=0)

# Normalize whitespace in ALL column names.
# CIC-IDS2017 contains headers such as " Label".
header.columns = header.columns.astype(str).str.strip()

columns = list(header.columns)

print("\nDataset:")
print("First file:", csv_files[0].name)
print("Columns:", len(columns))

# ------------------------------------------------------------
# Validate target column
# ------------------------------------------------------------

if TARGET_COLUMN not in columns:
    raise ValueError(
        f"Target column {TARGET_COLUMN!r} not found after "
        f"column-name normalization.\n"
        f"Available columns include:\n{columns[-10:]}"
    )

print(f"Target column {TARGET_COLUMN!r} confirmed.")

Looking for preprocessing specification:
/content/drive/MyDrive/CIC-Dataset-Analysis/results/final_dataset_selection/preprocessing_specification.csv
Exists: False

Found specification at:
- /content/results/final_dataset_selection/preprocessing_specification.csv

Using:
/content/results/final_dataset_selection/preprocessing_specification.csv

Preprocessing specification:


,parameter,value
0,dataset,CIC-IDS2017
1,target_column,Label
2,label_mode,multiclass
3,random_state,42
4,replace_infinite_with_nan,True
5,imputation,median_numeric_fit_on_train_only
6,scaling,StandardScaler_fit_on_train_only
7,remove_constant_features,True
8,remove_duplicates,True
9,oversampling,False



Resolved configuration:
Target column : 'Label'
Random state  : 42
Train fraction: 0.7
Validation    : 0.15
Test fraction : 0.15

Dataset:
First file: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Columns: 79
Target column 'Label' confirmed.


## 2. Establish the feature set from the raw schema

Constant features and suspicious/rejected features should be confirmed against the completed preprocessing artifacts before execution.

This cell automatically discovers candidate lists, but it **does not silently drop arbitrary 'suspicious' features**.

In [23]:
def read_feature_list_from_paths(paths):
    for p in paths:
        if p.exists():
            df = pd.read_csv(p)
            for c in ["feature", "feature_name", "column", "column_name"]:
                if c in df.columns:
                    return [str(x).strip() for x in df[c].dropna().tolist()]
    return []

constant_paths = [
    REPO / "results/cicids2017/05_preprocessing/constant_features.csv",
    REPO / "results/cicids2017/02_feature_data_quality/constant_features.csv",
]
suspicious_paths = [
    REPO / "results/cicids2017/05_preprocessing/suspicious_features.csv",
]

constant_candidates = read_feature_list_from_paths(constant_paths)
suspicious_candidates = read_feature_list_from_paths(suspicious_paths)

print("Constant candidates discovered:", len(constant_candidates))
print("Suspicious candidates discovered:", len(suspicious_candidates))
print("NOTE: suspicious candidates are NOT automatically removed.")

Constant candidates discovered: 0
Suspicious candidates discovered: 0
NOTE: suspicious candidates are NOT automatically removed.


### Explicit feature exclusions

Edit `EXPLICIT_DROP_FEATURES` only if the completed Notebook 05 preprocessing analysis says the feature should be removed.

This is deliberately explicit to prevent accidental feature deletion.

In [24]:
EXPLICIT_DROP_FEATURES = []  # Populate only from the completed preprocessing decision.
DROP_CONSTANTS = True

if EXPLICIT_DROP_FEATURES:
    unknown = set(EXPLICIT_DROP_FEATURES) - set(columns)
    if unknown:
        raise ValueError(f"Requested drop columns are not present: {sorted(unknown)}")

constant_drop = set(constant_candidates) if DROP_CONSTANTS else set()
drop_features = sorted((set(EXPLICIT_DROP_FEATURES) | constant_drop) - {TARGET_COLUMN})

print("Explicit drops:", EXPLICIT_DROP_FEATURES)
print("Constant drops:", len(constant_drop))
print("Total planned drops:", len(drop_features))

Explicit drops: []
Constant drops: 0
Total planned drops: 0


## 3. Read labels only and create a stratified split manifest

To avoid a massive feature matrix in memory, the notebook first creates a row-level manifest containing:

- source file
- source row index
- label
- split

The manifest is the basis for deterministic reconstruction of the final datasets.

> **Rare-class safeguard:** CIC-IDS2017 contains classes with extremely small support. A strict stratified 70/15/15 split is not statistically defensible for classes with only a handful of observations. The notebook therefore fails loudly rather than silently producing a misleading split. If this occurs, use the project-approved rare-class policy: keep the rare observations in training and report them as insufficient for independent validation/test estimation.


In [26]:
# ============================================================
# Build label manifest
# ============================================================

label_rows = []

for file in csv_files:
    print("Reading labels:", file.name)

    # --------------------------------------------------------
    # Read and normalize the header
    # --------------------------------------------------------

    raw_header = pd.read_csv(file, nrows=0)
    normalized_columns = (
        raw_header.columns
        .astype(str)
        .str.strip()
    )

    # Find the actual raw column corresponding to TARGET_COLUMN
    target_matches = np.where(
        normalized_columns == TARGET_COLUMN
    )[0]

    if len(target_matches) == 0:
        raise ValueError(
            f"Target column {TARGET_COLUMN!r} not found in "
            f"{file.name}.\n"
            f"Available columns include: "
            f"{list(normalized_columns[-10:])}"
        )

    target_index = int(target_matches[0])
    raw_target_column = raw_header.columns[target_index]

    print(
        f"  Target column: {raw_target_column!r} "
        f"→ normalized to {TARGET_COLUMN!r}"
    )

    # --------------------------------------------------------
    # Read target column by POSITION
    # --------------------------------------------------------
    # This avoids the leading-space issue in the raw CSV header.
    # The actual source column remains untouched.

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            file,
            usecols=[target_index],
            chunksize=200_000,
            low_memory=False
        )
    ):
        labels = (
            chunk.iloc[:, 0]
            .astype("string")
            .str.strip()
        )

        part = pd.DataFrame({
            "source_file": file.name,
            "source_row": np.arange(
                chunk_no * 200_000,
                chunk_no * 200_000 + len(chunk),
                dtype=np.int64
            ),
            "label": labels
        })

        label_rows.append(part)

# ------------------------------------------------------------
# Combine label manifest
# ------------------------------------------------------------

labels_df = pd.concat(
    label_rows,
    ignore_index=True
)

del label_rows
gc.collect()

print("\nManifest rows:", len(labels_df))
print("Classes:", labels_df["label"].nunique())

display(
    labels_df["label"]
    .value_counts(dropna=False)
    .head(20)
)

Reading labels: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Target column: ' Label' → normalized to 'Label'
Reading labels: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Target column: ' Label' → normalized to 'Label'
Reading labels: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Target column: ' Label' → normalized to 'Label'
Reading labels: Monday-WorkingHours.pcap_ISCX.csv
  Target column: ' Label' → normalized to 'Label'
Reading labels: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Target column: ' Label' → normalized to 'Label'
Reading labels: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Target column: ' Label' → normalized to 'Label'
Reading labels: Tuesday-WorkingHours.pcap_ISCX.csv
  Target column: ' Label' → normalized to 'Label'
Reading labels: Wednesday-workingHours.pcap_ISCX.csv
  Target column: ' Label' → normalized to 'Label'

Manifest rows: 2830743
Classes: 15


,count
label,
BENIGN,2273097
DoS Hulk,231073
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


In [27]:
# Stratified splitting is possible only when every class has enough observations.
# CIC-IDS2017 contains extremely rare classes, so we validate support first.
class_counts = labels_df["label"].value_counts()
print("Minimum class support:", class_counts.min())

if class_counts.min() < 4:
    raise ValueError(
        "A strict 70/15/15 stratified train/validation/test split cannot be "
        "performed safely because at least one class has fewer than 4 records. "
        "Do not silently duplicate or discard these rare observations. "
        "Use the rare-class-aware split strategy below."
    )

train_idx, temp_idx = train_test_split(
    np.arange(len(labels_df)),
    test_size=(VAL_FRAC + TEST_FRAC),
    stratify=labels_df["label"],
    random_state=RANDOM_STATE
)

temp_labels = labels_df.iloc[temp_idx]["label"]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=TEST_FRAC/(VAL_FRAC+TEST_FRAC),
    stratify=temp_labels,
    random_state=RANDOM_STATE
)

split = np.full(len(labels_df), "train", dtype=object)
split[val_idx] = "validation"
split[test_idx] = "test"
labels_df["split"] = split

manifest_path = OUT_DIR / "split_manifest.parquet"
labels_df.to_parquet(manifest_path, index=False)

print(labels_df["split"].value_counts())

Minimum class support: 11
split
train         1981520
test           424612
validation     424611
Name: count, dtype: int64


## 4. Materialize processed splits

The transformation is performed file-by-file and chunk-by-chunk.

Important:

- `inf -> NaN`
- numeric coercion is controlled
- no global scaling is performed before the split
- the notebook persists raw-ish cleaned features first
- train-fitted preprocessing parameters are then applied

If the complete dataset is too large for local memory, this stage should remain chunked.

In [28]:
# Build a cleaned tabular dataset in temporary Parquet partitions.
TMP_DIR = OUT_DIR / "tmp_clean"
TMP_DIR.mkdir(parents=True, exist_ok=True)

feature_columns = [c for c in columns if c != TARGET_COLUMN and c not in drop_features]

for file in csv_files:
    print("Processing:", file.name)
    source_manifest = labels_df[labels_df["source_file"] == file.name]
    row_offset = 0

    for chunk in pd.read_csv(file, chunksize=100_000, low_memory=False):
        chunk.columns = [c.strip() for c in chunk.columns]

        # Align schema.
        missing = set(feature_columns + [TARGET_COLUMN]) - set(chunk.columns)
        if missing:
            raise ValueError(f"{file.name}: missing columns {missing}")

        X = chunk[feature_columns].copy()
        y = chunk[TARGET_COLUMN].astype("string").str.strip()

        X = X.replace([np.inf, -np.inf], np.nan)

        # Numeric features only at this stage.
        for c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

        out = X
        out["label"] = y.values

        # Recover manifest rows using the source row range.
        rows_here = np.arange(row_offset, row_offset + len(chunk))
        split_lookup = source_manifest.set_index("source_row")["split"]
        out["split"] = pd.Series(rows_here, index=out.index).map(split_lookup).values
        out["source_file"] = file.name
        out["source_row"] = rows_here

        out.to_parquet(TMP_DIR / f"{file.stem}_{row_offset}.parquet", index=False)
        row_offset += len(chunk)

        del chunk, X, y, out
        gc.collect()

print("Temporary cleaned partitions:", len(list(TMP_DIR.glob("*.parquet"))))

Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Processing: Monday-WorkingHours.pcap_ISCX.csv
Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Processing: Wednesday-workingHours.pcap_ISCX.csv
Temporary cleaned partitions: 31


## 5. Fit imputer and scaler on TRAIN ONLY

This is the critical leakage-control step.

The imputer and scaler see **training rows only**. Validation and test data are transformed using those already-fitted parameters.

In [29]:
tmp_files = sorted(TMP_DIR.glob("*.parquet"))
if not tmp_files:
    raise FileNotFoundError("No temporary cleaned partitions were created.")

# For CIC-IDS2017 this is normally feasible on Colab; concatenate only the
# training rows for fitting. If memory becomes constrained, replace this
# section with an incremental scaler/imputer implementation.
train_parts = []
for p in tmp_files:
    df = pd.read_parquet(p, columns=feature_columns + ["label","split"])
    train_parts.append(df[df["split"] == "train"])

train_fit = pd.concat(train_parts, ignore_index=True)
del train_parts
gc.collect()

X_train_fit = train_fit[feature_columns].astype("float32")

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train_fit)

scaler = StandardScaler()
scaler.fit(X_train_imp)

joblib.dump(imputer, OUT_DIR / "imputer.joblib")
joblib.dump(scaler, OUT_DIR / "scaler.joblib")

with open(OUT_DIR / "feature_columns.json","w") as f:
    json.dump(feature_columns, f, indent=2)

print("Fitted on training rows:", len(train_fit))
print("Final feature count:", len(feature_columns))

Fitted on training rows: 1981520
Final feature count: 78


## 6. Encode labels and write final train/validation/test Parquet files

Label encoding is fitted globally from the known class vocabulary, but feature preprocessing remains train-fitted.

In [30]:
classes = sorted(labels_df["label"].dropna().astype(str).unique())
label_to_id = {label:i for i,label in enumerate(classes)}

with open(OUT_DIR / "label_mapping.json","w") as f:
    json.dump(label_to_id, f, indent=2)

# Recreate output datasets.
for split_name in ["train","validation","test"]:
    target = OUT_DIR / f"{split_name}.parquet"
    if target.exists():
        target.unlink()

for p in tmp_files:
    df = pd.read_parquet(p)
    split_name = df["split"].iloc[0] if df["split"].nunique()==1 else None

    if split_name is None:
        # Mixed partition: write each split separately.
        split_groups = df.groupby("split", sort=False)
    else:
        split_groups = [(split_name, df)]

    for s, part in split_groups:
        X = part[feature_columns].astype("float32")
        X = imputer.transform(X)
        X = scaler.transform(X).astype("float32")

        result = pd.DataFrame(X, columns=feature_columns)
        result["label"] = part["label"].astype(str).map(label_to_id).astype("int16").values

        # Append using pyarrow dataset semantics by writing partition files.
        part_dir = OUT_DIR / f"_parts_{s}"
        part_dir.mkdir(exist_ok=True)
        part_file = part_dir / f"{p.stem}_{s}.parquet"
        result.to_parquet(part_file, index=False)

        del X, result
        gc.collect()

print("Processed split parts created.")

Processed split parts created.


## 7. Consolidate split parts and validate

The consolidated Parquet files are the ML inputs. The temporary partition directories can be deleted after validation.

In [31]:
# Consolidate split parts using ParquetWriter instead of concatenating an entire
# split into RAM. This is important for the 2.8M-row CIC-IDS2017 dataset.
import pyarrow as pa
import pyarrow.parquet as pq

for split_name in ["train", "validation", "test"]:
    parts = sorted((OUT_DIR / f"_parts_{split_name}").glob("*.parquet"))
    if not parts:
        raise FileNotFoundError(f"No parts for {split_name}")

    target = OUT_DIR / f"{split_name}.parquet"
    if target.exists():
        target.unlink()

    writer = None
    total_written = 0

    try:
        for part_path in parts:
            part = pd.read_parquet(part_path)
            table = pa.Table.from_pandas(part, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(
                    target,
                    table.schema,
                    compression="snappy"
                )

            writer.write_table(table)
            total_written += len(part)

            del part, table
            gc.collect()
    finally:
        if writer is not None:
            writer.close()

    print(split_name, "rows:", total_written)

# Validate the final files without loading all columns.
summary_rows = []
for split_name in ["train", "validation", "test"]:
    df = pd.read_parquet(OUT_DIR / f"{split_name}.parquet", columns=["label"])
    for label, count in df["label"].value_counts().sort_index().items():
        summary_rows.append({
            "split": split_name,
            "label_id": int(label),
            "count": int(count)
        })
    print(split_name, "shape:", df.shape)
    del df
    gc.collect()

pd.DataFrame(summary_rows).to_csv(
    RESULTS_DIR / "final_class_distribution.csv",
    index=False
)

feature_manifest = pd.DataFrame({
    "feature": feature_columns,
    "dtype": "float32",
    "dropped": False
})
feature_manifest.to_csv(
    RESULTS_DIR / "final_feature_set.csv",
    index=False
)

print("Validation complete.")

train rows: 1981520
validation rows: 424611
test rows: 424612
train shape: (1981520, 1)
validation shape: (424611, 1)
test shape: (424612, 1)
Validation complete.


## 8. Provenance manifest

This records the transformation contract and output locations so the processed dataset is reproducible.

In [32]:
provenance = {
    "dataset": "CIC-IDS2017",
    "target_column": TARGET_COLUMN,
    "raw_files": [p.name for p in csv_files],
    "feature_count": len(feature_columns),
    "dropped_features": drop_features,
    "label_mapping": label_to_id,
    "split_ratios_requested": {"train":TRAIN_FRAC,"validation":VAL_FRAC,"test":TEST_FRAC},
    "random_state": RANDOM_STATE,
    "transformations": [
        "strip column-name whitespace",
        "replace +/-inf with NaN",
        "numeric coercion",
        "median imputation fitted on train only",
        "StandardScaler fitted on train only",
        "label encoding"
    ],
    "raw_data_modified": False
}
with open(OUT_DIR/"manifest.json","w") as f:
    json.dump(provenance,f,indent=2)

print(json.dumps(provenance, indent=2)[:5000])

{
  "dataset": "CIC-IDS2017",
  "target_column": "Label",
  "raw_files": [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv"
  ],
  "feature_count": 78,
  "dropped_features": [],
  "label_mapping": {
    "BENIGN": 0,
    "Bot": 1,
    "DDoS": 2,
    "DoS GoldenEye": 3,
    "DoS Hulk": 4,
    "DoS Slowhttptest": 5,
    "DoS slowloris": 6,
    "FTP-Patator": 7,
    "Heartbleed": 8,
    "Infiltration": 9,
    "PortScan": 10,
    "SSH-Patator": 11,
    "Web Attack \ufffd Brute Force": 12,
    "Web Attack \ufffd Sql Injection": 13,
    "Web Attack \ufffd XSS": 14
  },
  "split_ratios_requested": {
    "train": 0.7,
    "validat

## 9. Cleanup

After validation, remove temporary partition files. Keep:

```text
data/processed/cicids2017/
├── train.parquet
├── validation.parquet
├── test.parquet
├── feature_columns.json
├── label_mapping.json
├── imputer.joblib
├── scaler.joblib
├── split_manifest.parquet
└── manifest.json
```

The raw dataset remains unchanged.

In [34]:
import shutil

for d in OUT_DIR.glob("_parts_*"):
    shutil.rmtree(d, ignore_errors=True)
shutil.rmtree(TMP_DIR, ignore_errors=True)
gc.collect()

print("Temporary preprocessing partitions removed.")
print("Final processed artifacts:")
for p in sorted(OUT_DIR.iterdir()):
    print("-", p.name)

Temporary preprocessing partitions removed.
Final processed artifacts:
- feature_columns.json
- imputer.joblib
- label_mapping.json
- manifest.json
- scaler.joblib
- split_manifest.parquet
- test.parquet
- train.parquet
- validation.parquet


## Conclusion

The raw CIC-IDS2017 dataset has been converted into a reproducible ML-ready representation using a train-fitted preprocessing pipeline. No imputation or scaling parameters were learned from validation/test rows.

The resulting Parquet splits, feature specification, label mapping and preprocessing objects are now the canonical inputs for Notebook 15.